# 항공 데이터 전처리 (학원 PC 실행용)

학습은 집에서 코랩(`colab_run.ipynb`)으로, 전처리는 이 PC에서 이 노트북으로 합니다. 여기서 만든 결과는 `D:\LDH\ML\4ir`에만 저장되고 **드라이브로 올라가지 않습니다.** 합칠 때는 따로 요청합니다.

**쓰는 순서:** 아래 파일에서 전처리를 고친 뒤, 위에서부터 필요한 셀을 `Shift+Enter`로 실행합니다. 처음 실행할 때 커널은 `.venv`를 고릅니다.

| 고칠 내용 | 파일 |
|---|---|
| 통계와 원본 정리 | `categories/<카테고리>/pipeline.py` |
| 학습 입력(특성, 대상, 기간 나누기) | `categories/<카테고리>/models.py` |
| BTS 원본에서 가져올 열 | `prepare_data.py` |
| 사용할 첫 해 | `core/data.py`의 `START_YEAR` |

## 1. 환경 확인과 카테고리 선택

In [ ]:
%load_ext autoreload
%autoreload 2

import gc, importlib, os, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "run_all.py").exists():
    raise SystemExit(f"{ROOT} 에서 열렸습니다. D:\\LDH\\ML\\4ir 폴더를 VS Code로 열고 다시 실행하세요.")
if "google.colab" in sys.modules or any(word in str(ROOT) for word in ("드라이브", "Drive")):
    raise SystemExit("이 노트북은 학원 PC 전용입니다. 드라이브에서 실행하면 코랩 학습용 파일이 바뀝니다.")

from core.data import START_YEAR
from core.models import TEST, TRAIN, VAL, merge_summary
from core.registry import discover
from core.store import Store


def run(*args):
    """같은 파이썬으로 스크립트를 실행해 출력을 보여 주고, 실패하면 멈춘다. 고친 코드가 바로 반영되도록 별도 프로세스로 돌린다."""
    process = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1"})
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"실패: {args}")


ALL_SLUGS = [c["slug"] for c in discover()[0]]
print(f"파이썬 {sys.executable}")
print(f"사용할 첫 해 {START_YEAR}, CPU {os.cpu_count()}개, 디스크 여유 {shutil.disk_usage(ROOT).free / 1024**3:.0f}GB")
print("카테고리:", ALL_SLUGS)

In [ ]:
SLUGS = ["korea_demand"]  # 전처리를 고친 카테고리만 적는다. 전체는 ALL_SLUGS

unknown = set(SLUGS) - set(ALL_SLUGS)
if unknown:
    raise SystemExit(f"없는 카테고리: {unknown}")
print("이번 대상:", SLUGS)

## 2. BTS 원본을 parquet으로 변환 (필요할 때만)

`bts_data/`의 원본을 받았거나 `prepare_data.py`의 열을 바꿨을 때 실행합니다. 이미 변환된 파일은 건너뜁니다.

In [ ]:
run("prepare_data.py", str(START_YEAR))

## 3. 카테고리 전처리 실행

`SLUGS`의 `pipeline.py`를 실행해 PC의 `outputs/`를 다시 만듭니다. 전체를 돌리면 10분 이상 걸릴 수 있습니다.

In [ ]:
run("run_all.py", *SLUGS)
for slug in SLUGS:
    merge_summary(Store(slug))  # run_all 이 요약의 모델 결과 행을 지우므로 저장된 결과로 되살린다
print("완료:", SLUGS)

## 4. 큰 캐시 다시 만들기 (필요할 때만)

`us_flights`, `us_weather`, `world_traffic`의 **학습 입력을 바꿨을 때만** 실행합니다. 캐시가 이미 있으면 그대로 쓰기 때문에, 다시 만들지 않으면 옛 데이터로 점검됩니다. 수십 GB의 디스크와 긴 시간이 필요할 수 있습니다. 기본값은 빈 목록이라 아무것도 하지 않습니다.

In [ ]:
REBUILD = []  # 예: ["us_flights"]

for slug in REBUILD:
    if slug not in ("us_flights", "us_weather", "world_traffic"):
        raise SystemExit(f"{slug} 에는 큰 캐시가 없습니다")
    (importlib.import_module(f"categories.{slug}.models").ROOT / "meta.json").unlink(missing_ok=True)  # meta.json 이 없으면 build 가 다시 만든다
    run("-c", f"from categories.{slug}.models import build; build()")
print("다시 만든 캐시:", REBUILD)

## 5. 학습 입력 점검

전처리가 학습 입력에 어떻게 반영됐는지 표로 봅니다. 파일을 바꾸지 않으며, 결측률은 과제마다 최대 10만 행을 뽑아 계산합니다.

In [ ]:
def inspect_task(slug, t, sample_rows=100_000):
    row = {"카테고리": slug, "과제": t["task"]}
    try:
        d = t["load"]()
        split = np.asarray(d.split)
        used = np.flatnonzero(np.isin(split, (TRAIN, VAL, TEST)))
        rows = np.sort(np.random.default_rng(0).choice(used, min(sample_rows, len(used)), replace=False))
        missing = np.isnan(np.asarray(d.X[rows], dtype=np.float32)).mean(axis=0) * 100
        worst = int(missing.argmax())
        row.update({"학습 행": int((split == TRAIN).sum()), "검증 행": int((split == VAL).sum()), "평가 행": int((split == TEST).sum()),
                    "학습 기간": d.train_period, "평가 기간": d.test_period, "특성 수": len(d.columns), "결측 최대 특성": d.columns[worst],
                    "결측률(%)": round(float(missing[worst]), 1), "대상 결측(%)": round(float(np.isnan(np.asarray(d.y[rows], dtype=np.float64)).mean() * 100), 1)})
        del d
    except Exception as error:
        row["오류"] = f"{type(error).__name__}: {error}"
    gc.collect()
    return row


report = []
for slug in SLUGS:
    if (ROOT / "categories" / slug / "models.py").exists():
        report += [inspect_task(slug, t) for t in importlib.import_module(f"categories.{slug}.models").tasks()]
    else:
        print(f"{slug}: 학습 과제가 없어 건너뜁니다")
pd.set_option("display.max_rows", None, "display.max_columns", None)
pd.DataFrame(report)

## 끝난 뒤

- **결과 위치:** 이 PC의 `data/`와 `outputs/`에만 저장됩니다. `upload_to_drive.ps1`은 분리 기간 동안 막혀 있습니다.
- **합치기:** "전처리와 학습을 합쳐줘"라고 요청합니다. 양쪽 차이를 먼저 비교한 뒤 올리고, 바뀐 과제만 다시 학습합니다.